# SHIELD diffusion-limited analysis — from stored run to material properties

The standard SHIELD pipeline. It is valid when bulk diffusion is the slowest
step in the permeation chain (**diffusion-limited regime**), where:

- the steady flux follows Sieverts scaling J = Φ·√P/e → **permeability Φ**
  from the downstream rise slope,
- the transient obeys the Daynes–Barrer time lag τ = e²/(6D) → **diffusivity D**,
- **solubility** S = Φ/D.

The regime is an assumption to *verify*, not a given: flux must scale as √P.
[`regime_analysis.ipynb`](regime_analysis.ipynb) shows how to check it with a
pressure sweep. High upstream pressure and a hot sample favour DL; low
pressure, low temperature, or an oxidized surface push toward the
surface-limited regime, where every formula above silently returns the wrong
thing (see step 6).

This notebook walks the DL path on the uncoated-316L campaign:

1. **Browse** the stored runs in SHIELD-Data (`shield_data.catalogue`)
2. **Fetch** one by ID (`fetch_run`)
3. **Process** it (`process_run` — Φ, τ, D, S)
4. **Inspect** the run with the 2×2 overview figure
5. **Aggregate** a temperature series and fit **Arrhenius** trends
6. **Exclude** runs the time-lag method is not valid for

Prerequisite: the environment from the README's *Local development setup*
(`uv sync`, plus `uv pip install -e ../SHIELD-Data` for `shield_data`).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import shield_data as sd

from shield_toolbox import arrhenius, fetch_run, load_results, process_run
from shield_toolbox.plotting import plot_arrhenius, plot_run_overview

## 1. What runs exist?

The catalogue is one row per stored run — date, run type, furnace setpoint,
what sample was mounted, and which channels were recorded. It downloads once
from the `data-latest` release and is cached (inside a SHIELD-Data checkout
it reads `run_data/` directly).

In [ ]:
catalogue = sd.catalogue()
print(f"{len(catalogue)} stored runs")
columns = ["run_id", "date", "run_type", "furnace_setpoint", "substrate", "coating"]
permeation = catalogue[catalogue.run_type == "permeation_exp"]
permeation[columns].tail(8)

A quick look at what has been measured per sample and temperature:

In [ ]:
permeation.groupby(["substrate", "coating", "furnace_setpoint"]).size().rename(
    "runs"
).to_frame()

## 2. Fetch a run

`fetch_run` returns a `PermeationRun`: the raw recorded data (timestamps,
gauge voltages, valve events) plus metadata. Nothing is analysed yet.

In [ ]:
run = fetch_run("25.10.07_run_2_13h52")  # 316L steel, uncoated, 500 °C setpoint

print(f"run          : {run.run_id}")
print(f"duration     : {run.time_s[-1] / 3600:.1f} h ({len(run.time_s)} samples)")
print(f"gauges       : {sorted(run.gauge_voltages)}")
print(f"valve events : {run.valve_times_s}")
print(
    f"sample       : {run.metadata['run_info']['sample_substrate']} / "
    f"{run.metadata['run_info']['sample_coating']}"
)

## 3. Process it

`process_run` calibrates the Baratron voltages, restricts analysis to the
valid run window, and extracts the transport properties:

- **Φ (permeability)** from the steady-state downstream rise slope
  (Takaishi–Sensui corrected, uncertainty propagated)
- **τ (time lag)** from the fit's baseline crossing, timed from the
  loading-valve opening → **D = e²/(6τ)**
- **S = Φ/D**

The sample description and rig constants come from the metadata and the
versioned rig config — nothing to pass in for a stored run.

In [ ]:
processed = process_run(run)

print(
    f"sample T     : {processed.sample_temperature_K:.1f} K ({processed.temperature_source})"
)
print(f"P_up plateau : {processed.upstream_plateau.average_torr:.1f} Torr")
print(f"permeability : {processed.permeability:.2e}  H/(m·s·Pa^0.5)")
print(f"time lag     : {processed.time_lag_s:.0f} s")
print(f"diffusivity  : {processed.diffusivity_m2_per_s:.2e}  m²/s")
print(f"solubility   : {processed.solubility:.2e}  H/(m³·Pa^0.5)")

## 4. Inspect the run

Upstream pressure with the detected plateau, downstream rise with the fitted
slope, the analysis temperature, and the apparent permeability converging to
the steady-state value:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
plot_run_overview(processed, axes=axes)
fig.tight_layout()

## 5. A temperature series → Arrhenius fit

Process every uncoated-316L run of this campaign (300, 400, and 500 °C
setpoints), write the processed artifacts, and aggregate them. `write()`
stores each run under `<base>/<substrate>/<coating>/<run_id>/`; here the
base is a temporary directory — in day-to-day use you'd write to the
conventional `processed_runs/`.

In [ ]:
import tempfile
from pathlib import Path

RUN_IDS = [
    "25.10.10_run_1_08h38",
    "25.10.13_run_1_16h51",  # 300 °C
    "25.10.08_run_1_08h34",
    "25.10.08_run_2_20h05",
    "25.10.09_run_1_13h09",  # 400 °C
    "25.10.06_run_1_10h41",
    "25.10.07_run_1_08h16",
    "25.10.07_run_2_13h52",  # 500 °C
]

output_dir = Path(tempfile.mkdtemp())
for run_id in RUN_IDS:
    process_run(fetch_run(run_id)).write(output_dir)

results = load_results(output_dir, substrate="316L steel", coating="none")
results[
    [
        "run_id",
        "temperature_K",
        "permeability",
        "permeability_err",
        "time_lag_s",
        "diffusivity_m2_per_s",
        "solubility",
    ]
]

`arrhenius` runs an uncertainty-weighted fit of ln(property) vs 1/T — the
permeability is the most robust quantity, so fit it across all eight runs:

In [ ]:
fit = arrhenius(results, quantity="permeability")
print(f"activation energy : {fit.activation_energy_J_per_mol / 1000:.1f} kJ/mol")
print(f"pre-exponential   : {fit.pre_exponential:.2e}  H/(m·s·Pa^0.5)")

fig, ax = plt.subplots(figsize=(7, 5))
plot_arrhenius(results, fit=fit, ax=ax)
fig.tight_layout()

## 6. Regime hygiene: which runs may enter the D and S fits?

Look at the time lags in the table above. The two 555 K runs took
τ ≈ 1.7–2.8 **days**, giving apparent diffusivities two orders of magnitude
below the trend of the hotter runs. That is the classic symptom of surface
influence: when surface kinetics throttle the breakthrough, the time-lag
intercept no longer measures e²/(6D) — the delay reflects the surface, and it
grows with falling temperature much faster than diffusion alone would.

The steady flux is less distorted than the time lag near the crossover, but
not immune: surface throttling suppresses the low-temperature flux and
*flattens the apparent activation energy* of Φ. So refit all three
properties on the runs that are safely diffusion-limited — here, dropping the
555 K rows. Ideally, confirm the regime with a pressure sweep at the
temperature in question, as in [`regime_analysis.ipynb`](regime_analysis.ipynb).

In [ ]:
trusted = results[results.temperature_K > 600]

fit_phi = arrhenius(trusted, quantity="permeability")
print(
    f"Φ activation energy : {fit_phi.activation_energy_J_per_mol / 1000:.1f} kJ/mol "
    f"(was {fit.activation_energy_J_per_mol / 1000:.1f} with the 555 K runs in)"
)

fit_d = arrhenius(trusted, quantity="diffusivity")
print(f"D activation energy : {fit_d.activation_energy_J_per_mol / 1000:.1f} kJ/mol")
print(f"D pre-exponential   : {fit_d.pre_exponential:.2e}  m²/s")

fit_s = arrhenius(trusted, quantity="solubility")
print(f"S activation energy : {fit_s.activation_energy_J_per_mol / 1000:.1f} kJ/mol")
print(f"S pre-exponential   : {fit_s.pre_exponential:.2e}  H/(m³·Pa^0.5)")

## 7. Compare against literature — `h_transport_materials`

The [`h_transport_materials`](https://github.com/RemDelaporteMathurin/h-transport-materials) (htm)
database collects published Arrhenius parameters for hydrogen transport in
fusion-relevant materials. Plotting our fit on top of the literature spread
is the fastest credibility check on a campaign. htm plots in particle units
against 1/T (K⁻¹) — the same conventions as this toolbox, so the data overlay
directly with no conversion. (htm is a notebook-side dev dependency; the
analysis package itself never imports it.)

In [ ]:
import h_transport_materials as htm

htm_permeabilities = htm.permeabilities.filter(material="316l_steel").filter(
    isotope="h"
)

htm.plotting.plot(htm_permeabilities)
ax = plt.gca()
ax.errorbar(
    1 / results.temperature_K,
    results.permeability,
    yerr=results.permeability_err,
    fmt="o",
    markersize=8,
    color="black",
    capsize=3,
    label="SHIELD",
)
ax.plot(
    fit.fit_x_inverse_kK / 1000, fit.fit_y, "k-", lw=1.5, label="SHIELD Arrhenius fit"
)
ax.set_yscale("log")
ax.set_xlabel("1 / T (K$^{-1}$)")
ax.set_ylabel("Permeability (H m$^{-1}$ s$^{-1}$ Pa$^{-0.5}$)")
ax.legend(fontsize=7, loc="best")
plt.tight_layout()

Same pattern for the diffusivity — overlaying only the DL-trusted runs and
their refit from step 6 (`htm.solubilities` works identically for S):

In [ ]:
htm_diffusivities = htm.diffusivities.filter(material="316l_steel").filter(isotope="h")

htm.plotting.plot(htm_diffusivities)
ax = plt.gca()
ax.plot(
    1 / trusted.temperature_K,
    trusted.diffusivity_m2_per_s,
    "o",
    markersize=8,
    color="black",
    label="SHIELD (DL-trusted runs)",
)
ax.plot(
    fit_d.fit_x_inverse_kK / 1000,
    fit_d.fit_y,
    "k-",
    lw=1.5,
    label="SHIELD Arrhenius fit",
)
ax.set_yscale("log")
ax.set_xlabel("1 / T (K$^{-1}$)")
ax.set_ylabel("Diffusivity (m$^2$/s)")
ax.legend(fontsize=7, loc="best")
plt.tight_layout()

## Where to go next

- **Regime determination**: [`regime_analysis.ipynb`](regime_analysis.ipynb) — a pressure
  sweep at fixed temperature decides whether these formulas apply at all.
- **Surface-limited runs**: [`surface_limited_analysis.ipynb`](surface_limited_analysis.ipynb) —
  what to extract when they don't (K_d, K_r).
- **Command line**: `scripts/process_run.py` (single runs, saved figures) and
  `scripts/arrhenius.py` (campaign fits) do everything above in one call.
- **Fresh rig data**: `load_run(<run directory>)` loads a local run before it
  is uploaded — everything downstream is identical.
- **Rig utilities**: `load_furnace_log` / `furnace_temperature_offset` for
  Eurotherm furnace logs, and `fit_evacuation` for pump-down predictions —
  see the README's *Rig utilities* section.